In [2]:
import torch
from moabb.datasets import BNCI2014001, Cho2017, Lee2019_MI, Schirrmeister2017, PhysionetMI
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.preprocessing import LabelEncoder

from omegaconf import OmegaConf
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from moabb.utils import set_download_dir

from util import parse_args, set_determinism, set_run_dir

from hybrid_model import HybridModel
from hybrid_evaluation import HybridEvaluation
from hybrid_transform import HybridAggregateTransform
from hybrid_classifier import define_hybrid_clf

from paradigm import MotorImagery_

from pathlib import Path
import torchinfo

import numpy as np

from time import time

Tensorflow not install, you could not use those pipelines


In [3]:
# Define config path
config = OmegaConf.load(Path('/home/brunalopes/PycharmProjects/EEGHybrid/config/config.yaml'))

In [4]:
torch.set_num_threads(1)

cuda = (
    torch.cuda.is_available()
) 

In [5]:
dataset = BNCI2014001()
events = ["right_hand", "left_hand"]

paradigm = MotorImagery_(events=events, n_classes=len(events))

datasets = [dataset]
events = ["left_hand", "right_hand"]
n_classes = len(events)

# Load subject 8 and 5
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[5,8])
n_chans = X.shape[1]
input_window_samples = X.shape[2]
rpc = len(meta['session'].unique()) * len(meta['run'].unique())
le = LabelEncoder()
y=le.fit_transform(labels)

In [6]:
# Separate subject 8 and 5
subjects = meta.subject.values
mask8 = subjects == 8
X8, y8, meta8 = X[mask8], y[mask8], meta[meta['subject']==8]

# Subject 5
mask5 = subjects == 5
X5, y5, meta5 = X[mask5], y[mask5], meta[meta['subject']==5]

In [ ]:
model = HybridModel(1, 'EEGNet', n_chans, n_classes, input_window_samples, config=config, freeze='freeze')
path_model = Path('/home/brunalopes/PycharmProjects/EEGHybrid/Models/best_model_8-head-5_ea.pth')
model.load_state_dict(torch.load(path_model, weights_only=True))